# EFM cell-fate transition prediction

This tutorial shows how to predict perturbation-induced expression profiles with pretrained SFM and EFM. We start from a raw human normal myoblast dataset, preprocess it with the same package preprocessing stack used by the GRN tutorials, preserve the perturbation gene during HVG selection, and then run cell-fate transition prediction on the processed AnnData.

## 1. Setup

The input dataset is large, so keep the batch size modest unless you are running on a GPU with enough memory.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import scanpy as sc

start = Path.cwd().resolve()
repo_root = start if (start / "src").exists() else start.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Run this notebook from the scCAFM repository root or docs/.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.assets import load_vocab_json, resolve_model_assets
from src.config import load_yaml_config
from src.data import ScPreprocessor
from src.inference import cell_fate

## 2. Inputs

`imyoblast.h5ad` is treated as raw input here. The inference function itself assumes preprocessed input; the preprocessing step is explicit in this notebook. Like the GRN tutorials, `n_top_genes` controls the HVG budget and `max_length` controls tokenizer capacity.

In [ ]:
input_h5ad = Path("/data1021/xukaichen/data/CPP/imyoblast.h5ad")
model_source = repo_root / "assets"
config_path = repo_root / "configs" / "eval_cell_fate.yaml"

perturb_gene = "MYOD1"
layer_key = f"{perturb_gene}_cell_fate_transition"
generation_mode = "iterative_replace"  # or "one_forward"
use_kv_cache = True  # used only by iterative_replace
batch_size = 128
max_length = 1024
n_top_genes = 999 # remain one gene for perturbed gene

gene_key = None
species_key = "species"
disease_key = "disease"
cell_type_key = "cell_type"
cell_type_value = "S2A"

print(f"Input: {input_h5ad}")
print(f"Config: {config_path}")
print(f"Perturb gene: {perturb_gene}")

Input: /data1021/xukaichen/data/CPP/imyoblast.h5ad
Config: /data1021/xukaichen/scCAFM/configs/eval_cell_fate.yaml
Perturb gene: MYOD1


## 3. Load raw AnnData

The dataset file already contains the required condition metadata columns. We verify those columns here and leave the raw file metadata unchanged inside the notebook.

In [3]:
raw_adata = sc.read_h5ad(input_h5ad)

required_obs = [species_key, disease_key, cell_type_key]
missing_obs = [key for key in required_obs if key not in raw_adata.obs]
if missing_obs:
    raise KeyError(f"Missing required obs columns in {input_h5ad}: {missing_obs}")

print(f"Raw shape: {raw_adata.shape}")

raw_adata

Raw shape: (24991, 26017)


AnnData object with n_obs × n_vars = 24991 × 26017
    obs: 'species', 'disease', 'cell_type', 'annotation'

## 4. Preprocess AnnData

This follows normal HVG preprocessing, but preserves the perturbation gene so it remains available for the cell-fate simulation even if it is not selected as an HVG.

In [4]:
config = load_yaml_config(config_path)
assets = resolve_model_assets(
    model_source=model_source,
    require_model_weights=True,
    require_efm_config=True,
    require_efm_weights=True,
)
token_dict = load_vocab_json(assets.vocab)

preprocess_cfg = dict(config["data"]["preprocess"])
preprocess_cfg.pop("enabled", None)
preprocess_cfg["n_top_genes"] = n_top_genes
preprocess_cfg["preserve_gene_names"] = [perturb_gene]
preprocess_cfg["hvg_exclude_preserved_genes"] = True

preprocessor = ScPreprocessor(
    token_dict=token_dict,
    gene_key=gene_key,
    **preprocess_cfg,
)
processed_adata = preprocessor(raw_adata)

if processed_adata.n_vars > max_length - 1:
    raise ValueError(
        f"Processed AnnData has {processed_adata.n_vars} genes, but max_length={max_length} "
        "only leaves room for max_length - 1 gene tokens. Increase max_length or reduce n_top_genes."
    )

symbol_to_token = {
    str(row.gene_symbol).strip().upper(): int(row.token_index)
    for row in token_dict.itertuples(index=False)
    if pd.notna(row.gene_symbol) and str(row.gene_symbol).strip()
}
ensembl_to_token = {
    str(row.gene_id).strip().upper().split(".", 1)[0]: int(row.token_index)
    for row in token_dict.itertuples(index=False)
    if pd.notna(row.gene_id) and str(row.gene_id).strip()
}

def token_id_for_gene(gene):
    key = str(gene).strip().upper()
    return symbol_to_token.get(key, ensembl_to_token.get(key.split(".", 1)[0], -1))

gene_names = processed_adata.var_names if gene_key is None else processed_adata.var[gene_key].astype(str)
perturb_token_id = token_id_for_gene(perturb_gene)
active_token_ids = {token_id_for_gene(gene) for gene in gene_names}
if perturb_token_id < 0 or perturb_token_id not in active_token_ids:
    raise ValueError(
        f"Perturb gene {perturb_gene!r} is not present after preprocessing. "
        "Choose a gene present in the input and model vocabulary."
    )

print(f"Processed shape: {processed_adata.shape}")
processed_adata

Processed shape: (24991, 1000)


AnnData object with n_obs × n_vars = 24991 × 1000
    obs: 'species', 'disease', 'cell_type', 'annotation', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'preserved_gene', 'hvg_target_candidate', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'hvg_target_gene'
    uns: 'log1p'

In [5]:
inference_mask = processed_adata.obs[cell_type_key].astype(str) == cell_type_value
if not bool(inference_mask.any()):
    raise ValueError(f"No processed cells found with {cell_type_key} == {cell_type_value!r}.")

inference_adata = processed_adata[inference_mask].copy()

print(f"Full processed shape: {processed_adata.shape}")
print(f"{cell_type_value} inference shape: {inference_adata.shape}")
inference_adata

Full processed shape: (24991, 1000)
S2A inference shape: (512, 1000)


AnnData object with n_obs × n_vars = 512 × 1000
    obs: 'species', 'disease', 'cell_type', 'annotation', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'preserved_gene', 'hvg_target_candidate', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'hvg_target_gene'
    uns: 'log1p'

## 5. Prepare SFM + EFM inference

Full-dataset perturbation can be expensive, so this tutorial forwards the processed cells with `cell_type == "S1"`. `cell_fate.prepare` receives that already preprocessed subset and only builds tokenization, dataloader, and pretrained model state.

In [6]:
run = cell_fate.prepare(
    adata=inference_adata,
    model_source=model_source,
    config_path=config_path,
    batch_size=batch_size,
    max_length=max_length,
    gene_key=gene_key,
    species_key=species_key,
    disease_key=disease_key,
)

print(f"Inference AnnData shape: {run.adata.shape}")
print(f"SFM checkpoint: {run.sfm_checkpoint_file}")
print(f"EFM checkpoint: {run.efm_checkpoint_file}")

Inference AnnData shape: (512, 1000)
SFM checkpoint: /data1021/xukaichen/scCAFM/assets/models/sfm/sfm_model.safetensors
EFM checkpoint: /data1021/xukaichen/scCAFM/assets/models/efm/efm_model.safetensors


## 6. Predict perturbed expression profiles

For each cell, SFM defines a causal gene order. We set `MYOD1` to that cell's maximum active expression value, then EFM predicts expressions in the fixed SFM order. `iterative_replace` feeds each generated expression back before later genes; `one_forward` runs EFM once on the perturbed profile. Both modes keep the input gene IDs fixed and do not decode gene IDs from EFM's probability head. With `use_kv_cache=True`, iterative replacement freezes EFM's expression-dependent prefix token after the initial perturbed profile and reuses cached transformer K/V states.

In [7]:
run.config["cell_fate"]["generation_mode"] = generation_mode
run.config["cell_fate"]["use_kv_cache"] = use_kv_cache

predicted_adata = cell_fate.predict(
    run,
    perturb_gene=perturb_gene,
    layer_key=layer_key,
)

generated = predicted_adata.layers[layer_key]
print(f"Generated layer: {layer_key}")
print(f"Generated shape: {generated.shape}")
print(f"Generated dtype: {generated.dtype}")

Generated layer: MYOD1_cell_fate_transition
Generated shape: (512, 1000)
Generated dtype: float32


## 7. Inspect the prediction

The generated layer has the same cells and genes as `inference_adata`. The values are in the same expression scale as `processed_adata.X`.

In [8]:
gene_panel = [gene for gene in [perturb_gene, "MYOG", "MEF2C", "ACTA1"] if gene in predicted_adata.var_names]
if not gene_panel:
    gene_panel = predicted_adata.var_names[: min(4, predicted_adata.n_vars)].tolist()

original = np.asarray(predicted_adata[:, gene_panel].X)
generated_panel = np.asarray(predicted_adata[:, gene_panel].layers[layer_key])

summary = pd.DataFrame(
    {
        "gene": gene_panel,
        "original_mean": original.mean(axis=0),
        "generated_mean": generated_panel.mean(axis=0),
        "mean_delta": generated_panel.mean(axis=0) - original.mean(axis=0),
    }
)
summary

,gene,original_mean,generated_mean,mean_delta
0,MYOD1,0.130576,4.013322,3.882746
1,MYOG,0.033461,0.906773,0.873312
2,MEF2C,0.386077,0.826651,0.440574
3,ACTA1,0.053547,0.248615,0.195069


These results summarize the MYOD1 perturbation response in S2A cells. The table compares each selected gene's original expression with its generated expression after perturbation, making it easy to inspect whether MYOD1 and related myogenic markers shift in the expected direction.

## 8. Optional save

Set `save_output = True` to write the S1 inference AnnData plus generated layer.

In [ ]:
save_output = True
output_h5ad = repo_root / "results" / f"imyoblast_{perturb_gene}.h5ad"

if save_output:
    output_h5ad.parent.mkdir(parents=True, exist_ok=True)
    predicted_adata.write_h5ad(output_h5ad)
    print(f"Saved: {output_h5ad}")
else:
    print("Skipping write_h5ad. Set save_output = True to save the result.")

Saved: /data1021/xukaichen/scCAFM/results/imyoblast_MYOD1.h5ad


## Exercise

Try another myogenic regulator as `perturb_gene`, rerun preprocessing so the new gene is preserved, and compare the summary table. A common mistake is changing the perturbation gene only at prediction time; if the gene was not preserved during preprocessing, it may be absent from `processed_adata`.